In [195]:
import pandas as pd

df = pd.read_csv("http://114.207.245.181:13000/csv/Rossmann_Store_Sales.csv", low_memory=False)
df

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1
...,...,...,...,...,...,...,...,...,...
1017204,1111,2,2013-01-01,0,0,0,0,a,1
1017205,1112,2,2013-01-01,0,0,0,0,a,1
1017206,1113,2,2013-01-01,0,0,0,0,a,1
1017207,1114,2,2013-01-01,0,0,0,0,a,1


In [196]:
hangul_column_names = ["매장", "요일", "날짜", "매출", "고객수", "영업유무", "프로모션", "공휴일", "학교방학"]

In [197]:
t1 = dict(zip(df.columns, hangul_column_names))
t1

{'Store': '매장',
 'DayOfWeek': '요일',
 'Date': '날짜',
 'Sales': '매출',
 'Customers': '고객수',
 'Open': '영업유무',
 'Promo': '프로모션',
 'StateHoliday': '공휴일',
 'SchoolHoliday': '학교방학'}

In [198]:
df.rename(columns=t1, inplace=True)

In [199]:
# a => 일반공휴일
# b => 부활절
# c => 크리스마스
df['공휴일'].value_counts()
# df = df.astype({'공휴일': int})

공휴일
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64

In [200]:
df['날짜']  = pd.to_datetime(df['날짜'])
df['년']  = df['날짜'].dt.year
df['월']  = df['날짜'].dt.month
df['일']  = df['날짜'].dt.day


In [201]:
df.dtypes

매장               int64
요일               int64
날짜      datetime64[ns]
매출               int64
고객수              int64
영업유무             int64
프로모션             int64
공휴일             object
학교방학             int64
년                int32
월                int32
일                int32
dtype: object

In [ ]:
df1 = pd.get_dummies(
    df,
    columns=["공휴일"],
    drop_first=True,
    dtype=int
)
df1.head(10)

,매장,요일,날짜,매출,고객수,영업유무,프로모션,학교방학,년,월,일,공휴일_a,공휴일_b,공휴일_c
0,1,5,2015-07-31,5263,555,1,1,1,2015,7,31,0,0,0
1,2,5,2015-07-31,6064,625,1,1,1,2015,7,31,0,0,0
2,3,5,2015-07-31,8314,821,1,1,1,2015,7,31,0,0,0
3,4,5,2015-07-31,13995,1498,1,1,1,2015,7,31,0,0,0
4,5,5,2015-07-31,4822,559,1,1,1,2015,7,31,0,0,0
5,6,5,2015-07-31,5651,589,1,1,1,2015,7,31,0,0,0
6,7,5,2015-07-31,15344,1414,1,1,1,2015,7,31,0,0,0
7,8,5,2015-07-31,8492,833,1,1,1,2015,7,31,0,0,0
8,9,5,2015-07-31,8565,687,1,1,1,2015,7,31,0,0,0
9,10,5,2015-07-31,7185,681,1,1,1,2015,7,31,0,0,0


In [203]:
df1.dtypes

매장                int64
요일                int64
날짜       datetime64[ns]
매출                int64
고객수               int64
영업유무              int64
프로모션              int64
학교방학              int64
년                 int32
월                 int32
일                 int32
공휴일_a             int64
공휴일_b             int64
공휴일_c             int64
dtype: object

In [204]:
# 참고용 => 분포변경

import numpy as np

s = np.array([0, 100, 1000, 10000, 100000])
print("원본")
print(s)

print("분포변경")
g = np.log1p(s) # y = ln(1 + x)
print(g)

print("원본")
r = np.expm1(g) # y = e^x - 1
print(s)


원본
[     0    100   1000  10000 100000]
분포변경
[ 0.          4.61512052  6.90875478  9.21044037 11.51293546]
원본
[     0    100   1000  10000 100000]


In [246]:
y = np.log1p(df1['매출'].values)
x = df1.drop(columns=['날짜', '매출']).values
x.shape, y.shape

((1017209, 12), (1017209,))

In [247]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((813767, 12), (203442, 12), (813767,), (203442,))

In [248]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

x_train_scaled.shape, x_test_scaled.shape

((813767, 12), (203442, 12))

In [260]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(x_train_scaled.shape[1],)),    
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="linear")
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)
model.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_53 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_54 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_55 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [261]:
history = model.fit(
    x_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/20
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1.7329 - mae: 0.5661 - val_loss: 0.0528 - val_mae: 0.1715
Epoch 2/20
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0460 - mae: 0.1561 - val_loss: 0.0401 - val_mae: 0.1468
Epoch 3/20
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0390 - mae: 0.1418 - val_loss: 0.0363 - val_mae: 0.1392
Epoch 4/20
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0364 - mae: 0.1365 - val_loss: 0.0350 - val_mae: 0.1349
Epoch 5/20
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0353 - mae: 0.1339 - val_loss: 0.0345 - val_mae: 0.1343
Epoch 6/20
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0349 - mae: 0.1331 - val_loss: 0.0334 - val_mae: 0.1319
Epoch 7/20
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0346 - mae: 0.1326 - val_loss: 0.0337 - val_mae: 0.1328
Epoch 8/20
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0342 - mae: 0.1320 - val_loss: 0.0330 - val_mae: 0.1315
Epoch 9/20
1272/1272 ━━━━━━━━━━━━━━━━━━━

In [262]:
from sklearn.metrics import r2_score
y_pred_log = model.predict(x_test_scaled)
y_pred_orig = np.expm1(y_pred_log.flatten())
y_test_orig = np.expm1(y_test)
r2_score(y_test_orig, y_pred_orig)

6358/6358 ━━━━━━━━━━━━━━━━━━━━ 4s 646us/step


0.8708593049003115

In [268]:
y_xgb = y = np.log1p(df1['매출'].values)
x_xgb = df1.drop(columns=['날짜', '매출']).values
x_xgb.shape, y_xgb.shape

((1017209, 12), (1017209,))

In [272]:
from sklearn.model_selection import train_test_split

x_xgb_train, x_xgb_test, y_xgb_train, y_xgb_test = train_test_split(x_xgb, y_xgb, test_size=0.2)
x_xgb_train.shape, x_xgb_test.shape, y_xgb_train.shape, y_xgb_test.shape

((813767, 12), (203442, 12), (813767,), (203442,))

In [273]:
xgb_model = XGBRegressor()

xgb_model.fit(
    x_xgb_train,
    y_xgb_train,
)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [274]:
from sklearn.metrics import r2_score
y_xgb_pred_log = xgb_model.predict(x_xgb_test)
y_xgb_pred_orig = np.expm1(y_xgb_pred_log.flatten())
y_xgb_test_orig = np.expm1(y_xgb_test)
r2_score(y_xgb_test_orig, y_xgb_pred_orig)

0.9344132716026762